# 🎓 Smart Classroom AI: Integrated Backend
> **Status:** 🟢 System Ready  
> **Target Frontend:** [Vercel Deployment](https://smart-classroom-ui.vercel.app/)

---

### 🚀 Project Overview
This notebook serves as the high-performance backend for the Smart Classroom AI system. It orchestrates:
1. **Real-time STT:** Live lecture transcription using Whisper.
2. **RAG Engine:** Semantic search and Q&A over lecture materials using Qwen 2.5.
3. **Hybrid DB:** Local SQLite metadata + Supabase Vector Store (pgvector).
4. **Live Communication:** WebSocket-based chat and session management.

In [1]:
# Optional: Clean up invalid package metadata to silence pip warnings
import os
import shutil
from pathlib import Path

def cleanup_invalid_metadata():
    # Search for common dist-packages locations
    paths = ["/usr/local/lib/python3.12/dist-packages", "/usr/lib/python3/dist-packages"]

    for site_packages in paths:
        if os.path.exists(site_packages):
            # Target folders starting with '~' which indicate aborted installations
            invalid_dirs = [d for d in os.listdir(site_packages) if d.startswith("~")]
            for d in invalid_dirs:
                path = os.path.join(site_packages, d)
                try:
                    if os.path.isdir(path):
                        shutil.rmtree(path)
                    else:
                        os.remove(path)
                    print(f"✨ Cleaned up corrupt metadata: {d}")
                except Exception as e:
                    print(f"☑` Could not remove {d}: {e}")

cleanup_invalid_metadata()
print("\n✅ Metadata cleanup complete. You can now re-run your environment setup without those warnings.")


✅ Metadata cleanup complete. You can now re-run your environment setup without those warnings.


## 🛠️ Phase 1: Environment & Dependencies
*Cleaning and installing the CUDA-optimized ML stack.*

In [2]:
# %% =========================
# 1. CLEAN ENVIRONMENT
# =========================
# Silencing warnings by using --quiet and checking if package exists before uninstall
!pip -q uninstall -y torch torchvision torchaudio transformers accelerate bitsandbytes triton sentence-transformers peft 2>/dev/null || true

# Upgrade pip silently
!pip install -q --upgrade pip --disable-pip-version-check

# Helper for quiet installs
PIP_INSTALL = "pip install -q --disable-pip-version-check --no-warn-script-location"

# %% =========================
# 2. CORE ML / CUDA STACK
# =========================
!{PIP_INSTALL} --index-url https://download.pytorch.org/whl/cu121 \
  torch==2.4.1 \
  torchvision==0.19.1 \
  torchaudio==2.4.1

# %% =========================
# 3. TRANSFORMERS STACK
# =========================
!{PIP_INSTALL} \
  transformers==4.46.3 \
  accelerate==0.34.2 \
  bitsandbytes==0.43.3 \
  safetensors>=0.4.5 \
  sentencepiece \
  einops

# %% =========================
# 4. EMBEDDINGS / VECTOR DB
# =========================
!{PIP_INSTALL} \
  sentence-transformers==2.7.0 \
  faiss-cpu==1.8.0

# %% =========================
# 5. AUDIO / STT
# =========================
!{PIP_INSTALL} \
  faster-whisper==1.0.3 \
  webrtcvad==2.0.10 \
  soundfile

# %% =========================
# 6. BACKEND / API STACK
# =========================
!{PIP_INSTALL} \
  fastapi \
  uvicorn[standard] \
  python-socketio==5.11.0 \
  aiofiles \
  python-multipart \
  httpx

# %% =========================
# 7. DATABASE / AUTH / CLOUD
# =========================
!{PIP_INSTALL} \
  supabase \
  pgvector \
  openai \
  tiktoken \
  bcrypt \
  pyjwt \
  python-dotenv

# %% =========================
# 8. UTILITIES
# =========================
!{PIP_INSTALL} \
  numpy scipy pandas \
  pillow qrcode[pil] \
  python-docx reportlab PyPDF2 \
  orjson nest_asyncio pyngrok

# %% =========================
# 9. TRITON (MUST MATCH TORCH)
# =========================
!{PIP_INSTALL} triton==3.0.0

# %% =========================
# 10. IMPORTS
# =========================
import os, sys, json, time, uuid, hashlib, asyncio, logging, re, datetime
from pathlib import Path
import numpy as np
import torch

# Environment Check
print("\n✅ Environment setup complete (warnings suppressed)")
print("🔥 PyTorch version:", torch.__version__)
print("⚡ CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("🚀 GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Running on CPU")


✅ Environment setup complete (warnings suppressed)
🔥 PyTorch version: 2.4.1+cu121
⚡ CUDA available: True
🚀 GPU: Tesla T4


## 🔐 Phase 2: Security & Cloud Connectivity
*Injecting Supabase and ngrok credentials safely from Colab Secrets.*

In [3]:
import os
from google.colab import userdata
from google.colab.userdata import SecretNotFoundError

def get_secret_or_empty(key, default_value=""):
    try:
        return userdata.get(key).strip()
    except SecretNotFoundError:
        print(f"⚠️ Secret '{key}' not found. Using empty string or default.")
        return default_value
    except Exception as e:
        print(f"Error retrieving secret '{key}': {e}")
        return default_value

SUPABASE_URL = get_secret_or_empty("SUPABASE_URL")
SUPABASE_KEY = get_secret_or_empty("SUPABASE_KEY")
SUPABASE_SERVICE_KEY = get_secret_or_empty("SUPABASE_SERVICE_KEY")
NGROK_AUTH_TOKEN = get_secret_or_empty("NGROK_AUTH_TOKEN")
JWT_SECRET = get_secret_or_empty("JWT_SECRET", "smart-classroom-secret-2025")

# Sanity check
assert SUPABASE_URL and SUPABASE_KEY and SUPABASE_SERVICE_KEY, "⚠️ Fill all Supabase secrets first!"

# Expose to the process
os.environ["SUPABASE_URL"] = SUPABASE_URL
os.environ["SUPABASE_KEY"] = SUPABASE_KEY
os.environ["SUPABASE_SERVICE_KEY"] = SUPABASE_SERVICE_KEY
os.environ["NGROK_AUTH_TOKEN"] = NGROK_AUTH_TOKEN
os.environ["JWT_SECRET"] = JWT_SECRET

# 🗂 Create project folders


In [4]:
import os

PROJECT_ROOT = "/content"
DATA_ROOT = f"{PROJECT_ROOT}/smart_classroom"

folders = [
    f"{DATA_ROOT}/data",
    f"{DATA_ROOT}/audio",
    f"{DATA_ROOT}/cache",
    f"{DATA_ROOT}/templates",
    f"{DATA_ROOT}/logs",
    f"{DATA_ROOT}/exports",
]

for d in folders:
    os.makedirs(d, exist_ok=True)

print("✅ Folders ready")


✅ Folders ready


## 🗄️ Phase 3: Data Architecture
*Initializing the SQLite relational layer and connecting to the remote Vector Database.*

In [5]:
from pathlib import Path
import os
import json
import sqlite3
import threading
import hashlib
import datetime
import bcrypt
import numpy as np

from supabase import Client, create_client

DB_PATH = str(Path(DATA_ROOT) / "data" / "classroom.db")

class SupabaseVectorDB:
    """
    Handles the pgvector table in Supabase.
    Must be called after the Supabase schema / RPC are created.
    """
    def __init__(self):
        self.url = os.getenv("SUPABASE_URL")
        self.key = os.getenv("SUPABASE_SERVICE_KEY")
        self.client: Client | None = None
        self._connected = False
        self.embedding_dim = 384  # all-MiniLM-L6-v2 dimension

    def connect(self):
      if not self.url or not self.key:
          print("⚠️ Supabase config missing -> fallback to local SQLite only")
          return False
      try:
          self.client = create_client(self.url, self.key)
          self._connected = True
          print("✅ Supabase vector DB connected")
          return True
      except Exception as e:
          print(f"❌ Supabase connection error: {e}")
          return False



    def store_embeddings_batch(self, lecture_id: str, chunks: list[dict]) -> bool:
        """Insert many embeddings in a single RPC call."""
        if not self._connected or self.client is None:
            return False

        records = []
        for ch in chunks:
            records.append({
                "lecture_id": lecture_id,
                "chunk_idx": ch["chunk_idx"],
                "text": ch["text"][:2000],
                "embedding": ch["embedding"].tolist()
                if isinstance(ch["embedding"], np.ndarray) else ch["embedding"],
                "metadata": ch.get("metadata", {})
            })

        try:
            result = self.client.table("embeddings").insert(records).execute()
            return len(result.data) == len(records)
        except Exception as e:
            print(f"❌ Batch insert failed: {e}")
            return False

    def semantic_search(
        self,
        query_embedding: np.ndarray,
        lecture_id: str | None = None,
        top_k: int = 5,
        threshold: float = 0.6
    ) -> list[dict]:
        """
        Calls the Supabase RPC `match_embeddings` (you must add it
        in the Supabase SQL editor - see the code block below).
        """
        if not self._connected or self.client is None:
            return []

        try:
            query_vec = query_embedding.tolist()
            resp = self.client.rpc("match_embeddings", {
                "query_embedding": query_vec,
                "match_threshold": threshold,
                "match_count": top_k,
                "filter_lecture_id": lecture_id
            }).execute()
            return resp.data if resp.data else []
        except Exception as e:
            print(f"⚠️ Semantic search RPC failed -> fallback: {e}")
            return self._fallback_search(query_embedding, lecture_id, top_k, threshold)

    def _fallback_search(self, query_emb, lecture_id, top_k, thresh):
        """Local cosine-similarity search (slow, only for dev)."""
        try:
            q = self.client.table("embeddings").select("*")
            if lecture_id:
                q = q.eq("lecture_id", lecture_id)
            rows = q.limit(200).execute().data

            scored = []
            for r in rows:
                e = np.array(r["embedding"])
                sim = float(np.dot(query_emb, e) / (
                    np.linalg.norm(query_emb) * np.linalg.norm(e)
                ))
                if sim > thresh:
                    scored.append({**r, "similarity": sim})

            scored.sort(key=lambda x: x["similarity"], reverse=True)
            return scored[:top_k]
        except Exception as e:
            print(f"❌ Local fallback also failed: {e}")
            return []

    def get_stats(self):
        if not self._connected or self.client is None:
            return {"connected": False, "count": 0}
        try:
            cnt = self.client.table("embeddings").select("*", count="exact").limit(0).execute()
            return {
                "connected": True,
                "count": cnt.count or 0,
                "embedding_dim": self.embedding_dim
            }
        except Exception:
            return {"connected": True, "count": 0}


class SmartClassroomDB:
    def __init__(self, db_path: str = DB_PATH):
        self.db_path = db_path
        self.conn = sqlite3.connect(db_path, check_same_thread=False)
        self.conn.row_factory = sqlite3.Row
        self._lock = threading.Lock()
        self._create_tables()
        self._seed_demo_user()

        # Initialize the Supabase vector DB
        self.vector_db = SupabaseVectorDB()
        self.vector_db.connect()

    def _create_tables(self):
        with self._lock:
            self.conn.executescript("""
            CREATE TABLE IF NOT EXISTS users (
                id TEXT PRIMARY KEY,
                username TEXT UNIQUE NOT NULL,
                password_hash TEXT NOT NULL,
                email TEXT,
                role TEXT CHECK(role IN ('teacher','student','admin')) NOT NULL,
                created_at TEXT,
                last_login TEXT
            );
            CREATE TABLE IF NOT EXISTS lectures (
                id TEXT PRIMARY KEY,
                title TEXT NOT NULL,
                course TEXT NOT NULL,
                week INTEGER,
                content TEXT,
                topics TEXT,
                outcomes TEXT,
                chunk_count INTEGER,
                uploaded_at TEXT,
                file_type TEXT,
                original_filename TEXT
            );
            CREATE TABLE IF NOT EXISTS lecture_chunks_local (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                lecture_id TEXT NOT NULL,
                chunk_idx INTEGER NOT NULL,
                text TEXT NOT NULL,
                faiss_idx INTEGER,
                embedding BLOB,
                FOREIGN KEY (lecture_id) REFERENCES lectures(id)
            );
            CREATE TABLE IF NOT EXISTS questions (
                id TEXT PRIMARY KEY,
                lecture_id TEXT NOT NULL,
                text TEXT NOT NULL,
                bloom TEXT,
                qtype TEXT,
                options TEXT,
                correct TEXT,
                explanation TEXT,
                difficulty REAL,
                confidence REAL,
                created_at TEXT
            );
            CREATE TABLE IF NOT EXISTS attendance_sessions (
                id TEXT PRIMARY KEY,
                lecture_id TEXT,
                start_time TEXT,
                end_time TEXT,
                created_at TEXT
            );
            CREATE TABLE IF NOT EXISTS attendance_records (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                session_id TEXT NOT NULL,
                student_name TEXT NOT NULL,
                checkin_time TEXT NOT NULL,
                FOREIGN KEY (session_id) REFERENCES attendance_sessions(id)
            );
            CREATE TABLE IF NOT EXISTS reminders (
                id TEXT PRIMARY KEY,
                title TEXT NOT NULL,
                description TEXT,
                due_date TEXT,
                due_time TEXT,
                category TEXT DEFAULT 'general',
                priority TEXT DEFAULT 'medium',
                created_at TEXT,
                completed INTEGER DEFAULT 0
            );
            CREATE TABLE IF NOT EXISTS calendar_events (
                id TEXT PRIMARY KEY,
                title TEXT NOT NULL,
                description TEXT,
                event_date TEXT NOT NULL,
                event_time TEXT,
                end_time TEXT,
                event_type TEXT DEFAULT 'class',
                course TEXT,
                color TEXT DEFAULT '#0ea5e9',
                created_at TEXT
            );
            CREATE TABLE IF NOT EXISTS live_sessions (
                id TEXT PRIMARY KEY,
                title TEXT,
                course TEXT,
                stt_model TEXT,
                llm_model TEXT,
                tts_model TEXT,
                started_at TEXT,
                ended_at TEXT
            );
            CREATE TABLE IF NOT EXISTS live_insights (
                id TEXT PRIMARY KEY,
                session_id TEXT,
                ts REAL,
                transcript TEXT,
                summary TEXT,
                key_concepts TEXT,
                difficulty INTEGER,
                engagement REAL,
                action_items TEXT,
                teacher_tip TEXT,
                tip_audio TEXT,
                processing_time_ms REAL,
                created_at TEXT
            );
            CREATE TABLE IF NOT EXISTS live_chat (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                session_id TEXT,
                ts REAL,
                message TEXT,
                response TEXT,
                created_at TEXT
            );
            """)
            self.conn.commit()

    def _seed_demo_user(self):
        """Creates a demo teacher account if none exists."""
        cur = self.conn.execute("SELECT 1 FROM users WHERE username='teacher'")
        if cur.fetchone():
            return

        pw_hash = bcrypt.hashpw(b"password123", bcrypt.gensalt()).decode()
        uid = hashlib.sha256(f"teacher_{datetime.datetime.now()}".encode()).hexdigest()[:16]
        self.conn.execute(
            "INSERT INTO users (id,username,password_hash,role,created_at) VALUES (?,?,?,?,?)",
            (uid, "teacher", pw_hash, "teacher", datetime.datetime.now().isoformat())
        )
        self.conn.commit()
        print("✅ Demo user: teacher / password123")

    def register_user(self, username, password, email=None, role="teacher"):
        if role not in ("teacher", "student", "admin"):
            raise ValueError("Invalid role")
        if self.get_user_by_username(username):
            raise ValueError("Username already exists")

        pw_hash = bcrypt.hashpw(password.encode(), bcrypt.gensalt()).decode()
        uid = hashlib.sha256(f"{username}_{datetime.datetime.now()}".encode()).hexdigest()[:16]
        now = datetime.datetime.now().isoformat()
        self.conn.execute(
            "INSERT INTO users (id,username,password_hash,email,role,created_at) VALUES (?,?,?,?,?,?)",
            (uid, username, pw_hash, email, role, now)
        )
        self.conn.commit()
        return {"id": uid, "username": username, "email": email, "role": role}

    def get_user_by_username(self, username):
        cur = self.conn.execute("SELECT * FROM users WHERE username=?", (username,))
        row = cur.fetchone()
        return dict(row) if row else None

    def update_last_login(self, username):
        self.conn.execute(
            "UPDATE users SET last_login=? WHERE username=?",
            (datetime.datetime.now().isoformat(), username)
        )
        self.conn.commit()

    def save_lecture(self, lec: dict):
        with self._lock:
            self.conn.execute(
                """
                INSERT OR REPLACE INTO lectures
                (id,title,course,week,content,topics,outcomes,chunk_count,uploaded_at,file_type,original_filename)
                VALUES (?,?,?,?,?,?,?,?,?,?,?)
                """,
                (
                    lec["id"],
                    lec["title"],
                    lec["course"],
                    lec.get("week"),
                    lec.get("content", ""),
                    json.dumps(lec.get("topics", [])),
                    json.dumps(lec.get("outcomes", [])),
                    int(lec.get("chunk_count", 0)),
                    lec.get("uploaded_at", datetime.datetime.now().isoformat()),
                    lec.get("file_type", "text"),
                    lec.get("original_filename", "")
                )
            )
            self.conn.commit()

        def save_lecture_chunks(self, lecture_id, metas):
            """Persist chunks both locally (SQLite) and remotely (Supabase)."""
            with self._lock:
                for m in metas:
                    # Convert embedding to JSON/BLOB for SQLite if present
                    embed_blob = json.dumps(m.get("embedding", [])) if m.get("embedding") is not None else None
                    self.conn.execute(
                        "INSERT INTO lecture_chunks_local (lecture_id,chunk_idx,text,faiss_idx,embedding) VALUES (?,?,?,?,?)",
                        (lecture_id, m["chunk_idx"], m["text"], m["faiss_idx"], embed_blob)
                    )
                self.conn.commit()

                # Send the full 'metas' (which includes 'embedding') to Supabase
                if self.vector_db._connected:
                    self.vector_db.store_embeddings_batch(lecture_id, metas)


    def get_lecture_detail(self, lecture_id):
      with self._lock:
          lec = self.conn.execute("SELECT * FROM lectures WHERE id=?", (lecture_id,)).fetchone()
          if not lec:
              return None

          chunks = self.conn.execute(
              "SELECT chunk_idx,text FROM lecture_chunks_local WHERE lecture_id=? ORDER BY chunk_idx",
              (lecture_id,)
          ).fetchall()

          lecture = dict(lec)
          lecture["chunks"] = [c["text"] for c in chunks]
          lecture["topics"] = json.loads(lecture["topics"]) if lecture["topics"] else []
          lecture["outcomes"] = json.loads(lecture["outcomes"]) if lecture["outcomes"] else []
          return lecture

    def list_lectures(self):
        with self._lock:
            rows = self.conn.execute(
                """
                SELECT id, title, course, week, chunk_count, uploaded_at, file_type, original_filename
                FROM lectures
                ORDER BY uploaded_at DESC
                """
            ).fetchall()
            return [dict(row) for row in rows]

    def delete_lecture(self, lecture_id: str):
        """Delete lecture and its chunks (teacher only)."""
        with self._lock:
            # Delete local chunks
            self.conn.execute("DELETE FROM lecture_chunks_local WHERE lecture_id=?", (lecture_id,))
            # Delete lecture record
            self.conn.execute("DELETE FROM lectures WHERE id=?", (lecture_id,))
            self.conn.commit()

        # Delete from Supabase vector store if connected
        if self.vector_db and self.vector_db._connected and self.vector_db.client:
            try:
                self.vector_db.client.table("embeddings").delete().eq("lecture_id", lecture_id).execute()
                print(f"🗑️ Deleted embeddings from Supabase for {lecture_id}")
            except Exception as e:
                print(f"⚠️ Could not delete from Supabase vector store: {e}")

    def get_questions_by_lecture(self, lecture_id: str):
        """Get questions for a specific lecture."""
        with self._lock:
            rows = self.conn.execute(
                "SELECT * FROM questions WHERE lecture_id=? ORDER BY created_at DESC",
                (lecture_id,)
            ).fetchall()
            return [dict(row) for row in rows]

    def get_all_questions(self, limit: int = 50):
        """Get all questions."""
        with self._lock:
            rows = self.conn.execute(
                "SELECT * FROM questions ORDER BY created_at DESC LIMIT ?",
                (limit,)
            ).fetchall()
            return [dict(row) for row in rows]



    def db_stats(self):
        tables = [
            "lectures", "questions", "users", "attendance_sessions",
            "attendance_records", "reminders", "calendar_events",
            "live_sessions", "live_insights", "live_chat"
        ]
        counts = {}
        for t in tables:
            try:
                cnt = self.conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
            except Exception:
                cnt = 0
            counts[t] = cnt

        size_kb = os.path.getsize(self.db_path) / 1024 if os.path.exists(self.db_path) else 0
        vector_stats = self.vector_db.get_stats() if hasattr(self, "vector_db") else {"connected": False}
        return {"tables": counts, "size_kb": round(size_kb, 1), "vector_db": vector_stats}


# Initialize the DB (creates SQLite + connects to Supabase)
print("\n📦 Initializing database...")
db = SmartClassroomDB(DB_PATH)
print("✅ DB ready - stats:", db.db_stats())



📦 Initializing database...
✅ Supabase vector DB connected
✅ DB ready - stats: {'tables': {'lectures': 0, 'questions': 0, 'users': 3, 'attendance_sessions': 0, 'attendance_records': 0, 'reminders': 0, 'calendar_events': 0, 'live_sessions': 0, 'live_insights': 0, 'live_chat': 0}, 'size_kb': 96.0, 'vector_db': {'connected': True, 'count': 0, 'embedding_dim': 384}}


## 🧠 Phase 4: AI Inference Engine
*Loading Qwen 2.5-3B-Instruct with 4-bit quantization for rapid response.*

In [6]:
import gc
import threading
import torch


class LLMEngine:
    """
    Handles loading and generation for Qwen 2.5-3B-Instruct.
    4-bit quantization is used when a CUDA GPU is present.
    """
    def __init__(
        self,
        model_name: str = "Qwen/Qwen2.5-3B-Instruct",
        max_seq_len: int = 4096,
        use_4bit: bool = True
    ):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model_name = model_name
        self.max_seq_len = max_seq_len
        self.use_4bit = use_4bit and torch.cuda.is_available()
        self.tokenizer = None
        self.model = None
        self._loaded = False
        self._loading_error = None

    def _check_4bit_available(self) -> bool:
        if not torch.cuda.is_available():
            return False
        try:
            from transformers import BitsAndBytesConfig
            _ = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            )
            return True
        except Exception as e:
            print(f"⚠️ 4-bit unavailable: {e}")
            return False

    def load(self, force_4bit: bool | None = None):
        if self._loaded:
            return True

        try_4bit = force_4bit if force_4bit is not None else self.use_4bit
        if try_4bit:
            try_4bit = self._check_4bit_available()

        print(f"🔄 Loading {self.model_name} on {self.device} (4-bit={try_4bit})")

        try:
            from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

            self.tokenizer = AutoTokenizer.from_pretrained(
                self.model_name,
                trust_remote_code=True,
                use_fast=True
            )

            if not self.tokenizer.pad_token:
                self.tokenizer.pad_token = self.tokenizer.eos_token

            kwargs = {
                "device_map": "auto",
                "trust_remote_code": True,
                "torch_dtype": torch.float16,
            }

            if try_4bit:
                kwargs["quantization_config"] = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_use_double_quant=True,
                )
                kwargs.pop("torch_dtype", None)

            self.model = AutoModelForCausalLM.from_pretrained(self.model_name, **kwargs)
            self.model.eval()
            self._loaded = True
            print("✅ Model loaded")
            return True

        except Exception as e:
            self._loading_error = str(e)
            print(f"❌ Model load error: {e}")
            if try_4bit and "bitsandbytes" in str(e).lower():
                print("🔄 Retrying without 4-bit...")
                return self.load(force_4bit=False)
            return False

    @torch.inference_mode()
    def stream_generate(
        self,
        messages: list[dict],
        max_new_tokens: int = 512,
        temperature: float = 0.7
    ):
        if not self._loaded:
            raise RuntimeError("Model not loaded")

        from transformers import TextIteratorStreamer

        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_seq_len,
            padding=True
        ).to(self.device)

        streamer = TextIteratorStreamer(
            self.tokenizer,
            skip_prompt=True,
            skip_special_tokens=True
        )

        gen_kwargs = {
            **inputs,
            "streamer": streamer,
            "max_new_tokens": max_new_tokens,
            "do_sample": True,
            "temperature": max(temperature, 0.01),
            "top_p": 0.9,
            "repetition_penalty": 1.1,
            "pad_token_id": self.tokenizer.pad_token_id,
            "eos_token_id": self.tokenizer.eos_token_id,
        }

        thread = threading.Thread(target=self.model.generate, kwargs=gen_kwargs)
        thread.start()

        for new_text in streamer:
            yield new_text

        thread.join()

    def generate_sync(
        self,
        messages: list[dict],
        max_new_tokens: int = 256,
        temperature: float = 0.7
    ) -> str:
        out = ""
        for chunk in self.stream_generate(
            messages,
            max_new_tokens=max_new_tokens,
            temperature=temperature
        ):
            out += chunk
        return out.strip()

    def unload(self):
        if self.model:
            del self.model
        if self.tokenizer:
            del self.tokenizer

        self.model = None
        self.tokenizer = None
        self._loaded = False

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print("🗑️ Model unloaded")

    @property
    def is_loaded(self):
        return self._loaded

    @property
    def error(self):
        return self._loading_error


class LLMAdapter:
    """Thin wrapper that the rest of the code talks to."""
    def __init__(
        self,
        model_name: str = "Qwen/Qwen2.5-3B-Instruct",
        force_4bit: bool = True
    ):
        self.engine = LLMEngine(model_name=model_name, use_4bit=force_4bit)
        self._ready = self.engine.load()
        if not self._ready:
            print(f"⚠️ LLM init failed: {self.engine.error}")

    def generate_chat(
        self,
        messages: list[dict],
        max_new_tokens: int = 256,
        temperature: float = 0.7
    ) -> str:
        if not self._ready:
            return "[LLM unavailable]"
        return self.engine.generate_sync(messages, max_new_tokens, temperature)

    def stream_generate(
        self,
        messages: list[dict],
        max_new_tokens: int = 512,
        temperature: float = 0.7
    ):
        if not self._ready:
            yield "[LLM unavailable]"
            return
        yield from self.engine.stream_generate(messages, max_new_tokens, temperature)

    def generate_with_context(
        self,
        query: str,
        context_chunks: list[str],
        system_prompt: str | None = None,
        max_new_tokens: int = 512,
        temperature: float = 0.7
    ) -> str:
        """Convenient function used by the RAG engine."""
        if not self._ready:
            return "[LLM unavailable]"

        context = "\n\n".join(
            [f"[{i + 1}] {c}" for i, c in enumerate(context_chunks[:5])]
        )

        messages = [
            {
                "role": "system",
                "content": system_prompt or (
                    "You are a helpful teaching assistant. "
                    "Answer ONLY using the provided lecture excerpts. "
                    "Cite chunk numbers when you reference them."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Based on the following lecture excerpts:\n\n{context}\n\n"
                    f"Question: {query}\n\n"
                    "Provide a concise answer."
                ),
            },
        ]
        return self.generate_chat(messages, max_new_tokens, temperature)

    @property
    def is_ready(self):
        return self._ready and self.engine.is_loaded
# Initialise a **shared** LLM that will be reused by every request
print("\n🧠 Loading shared LLM …")
shared_llm = LLMAdapter(model_name="Qwen/Qwen2.5-3B-Instruct", force_4bit=True)
print(f"✅ LLM ready? {shared_llm.is_ready}")



🧠 Loading shared LLM …
🔄 Loading Qwen/Qwen2.5-3B-Instruct on cuda (4-bit=True)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Model loaded
✅ LLM ready? True


#Supabase RAG Engine

In [7]:
#Supabase RAG Engine

import json
import re
import uuid
import datetime
import numpy as np

from sentence_transformers import SentenceTransformer
from fastapi import HTTPException


class SupabaseRAGEngine:
    """
    Complete RAG pipeline:
        * generate (or load) sentence-transformer embeddings,
        * store them in Supabase,
        * retrieve top-k relevant chunks for a query,
        * let the LLM answer using the retrieved context.
    """
    def __init__(
        self,
        llm_adapter: LLMAdapter | None = None,
        vector_db: SupabaseVectorDB | None = None
    ):
        self.llm = llm_adapter
        self.vector_db = vector_db
        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")
        self.lecture_chunks: dict[str, list[str]] = {}
        self.lecture_meta: dict[str, dict] = {}
        self._load_existing_lectures()

    def _load_existing_lectures(self):
        """Populate `lecture_chunks` from the SQLite cache at startup."""
        try:
            for lec in db.list_lectures():
                detail = db.get_lecture_detail(lec["id"])
                if detail and detail.get("chunks"):
                    self.lecture_chunks[lec["id"]] = detail["chunks"]
                    self.lecture_meta[lec["id"]] = {
                        "title": detail.get("title", ""),
                        "course": detail.get("course", ""),
                        "topics": detail.get("topics", [])
                    }
            print(f"🔹 Loaded {len(self.lecture_chunks)} lectures from SQLite")
        except Exception as e:
            print(f"⚠️ Could not load lectures: {e}")

    def _embed(self, text: str) -> np.ndarray:
        """Fast single-sentence embedding (normalized)."""
        return self.embedder.encode(text, normalize_embeddings=True)

    def search_lecture(
        self,
        lecture_id: str | None,
        query: str,
        top_k: int = 5,
        threshold: float = 0.6
    ) -> list[dict]:
        """
        Returns a list of dicts:
            {lecture_id, chunk_idx, text, similarity, ...}
        """
        q_emb = self._embed(query)

        # Supabase vector search
        if self.vector_db and self.vector_db._connected:
            results = self.vector_db.semantic_search(
                query_embedding=q_emb,
                lecture_id=lecture_id,
                top_k=top_k,
                threshold=threshold
            )
            if results:
                return results

        # Local fallback
        candidates = []
        if lecture_id:
            chunks = self.lecture_chunks.get(lecture_id, [])
            for i, txt in enumerate(chunks):
                overlap = len(set(query.lower().split()) & set(txt.lower().split())) / max(1, len(query.split()))
                if overlap > 0.1:
                    candidates.append({
                        "lecture_id": lecture_id,
                        "chunk_idx": i,
                        "text": txt,
                        "similarity": overlap
                    })
        else:
            for lid, chunks in self.lecture_chunks.items():
                for i, txt in enumerate(chunks):
                    overlap = len(set(query.lower().split()) & set(txt.lower().split())) / max(1, len(query.split()))
                    if overlap > 0.1:
                        candidates.append({
                            "lecture_id": lid,
                            "chunk_idx": i,
                            "text": txt,
                            "similarity": overlap
                        })

        candidates.sort(key=lambda x: x["similarity"], reverse=True)
        return candidates[:top_k]

    def ask_with_context(
        self,
        question: str,
        lecture_id: str | None = None,
        top_k: int = 5
    ) -> dict:
        """
        1. Retrieve relevant chunks
        2. Build a prompt with those chunks
        3. Call the LLM
        4. Return answer + source metadata + metrics
        """
        if not self.llm or not self.llm.is_ready:
            return {
                "answer": "LLM not available - try again later.",
                "sources": [],
                "context_used": 0
            }

        results = self.search_lecture(lecture_id, question, top_k=top_k)
        if not results:
            return {
                "answer": "", # Return empty for fallback handling
                "sources": [],
                "context_used": 0
            }

        context_chunks = [r["text"] for r in results]

        answer = self.llm.generate_with_context(
            query=question,
            context_chunks=context_chunks,
            system_prompt=(
                "You are a knowledgeable teaching assistant. "
                "Answer ONLY using the supplied lecture excerpts. "
                "Cite chunk numbers."
            ),
            max_new_tokens=500,
            temperature=0.7
        )

        sources = []
        for r in results:
            meta = self.lecture_meta.get(r["lecture_id"], {})
            sources.append({
                "lecture_id": r["lecture_id"],
                "lecture_title": meta.get("title", ""),
                "chunk_idx": r["chunk_idx"],
                "similarity": round(r.get("similarity", 0), 3),
                "preview": r["text"][:150] + "..."
            })

        return {
            "answer": answer,
            "sources": sources,
            "context_used": len(context_chunks),
            "retrieval_method": (
                "supabase_vector"
                if (self.vector_db and self.vector_db._connected)
                else "local_fallback"
            )
        }

    def generate_questions(
        self,
        lecture_id: str,
        bloom_level: str = "understand",
        qtype: str = "mcq",
        count: int = 5
    ) -> list:
        """Generate quiz questions using LLM."""
        if not self.llm or not self.llm.is_ready:
            return []

        # Get lecture content
        lec = db.get_lecture_detail(lecture_id)
        if not lec:
            return []

        content = lec.get("content", "")[:3000]  # Limit content

        prompt = [
            {
                "role": "system",
                "content": f"""You are an educational assessment expert. Create {count} {qtype} questions at {bloom_level} level.
                Return ONLY a JSON array of objects with fields: text, correct_answer, options (array), explanation, bloom_level, qtype.
                For mcq, include 4 options."""
            },
            {
                "role": "user",
                "content": f"Lecture Title: {lec['title']}\n\nContent: {content}"
            }
        ]

        try:
            resp = self.llm.generate_chat(prompt, max_new_tokens=800, temperature=0.7)
            # Extract JSON from response
            import re
            json_match = re.search(r'\[.*\]', resp, re.DOTALL)
            if json_match:
                questions = json.loads(json_match.group())
                # Add metadata
                for q in questions:
                    q["id"] = str(uuid.uuid4())[:16]
                    q["lecture_id"] = lecture_id
                    q["created_at"] = datetime.datetime.now().isoformat()
                    # Store in DB
                    self._save_question(q)
                return questions
        except Exception as e:
            print(f"Question generation failed: {e}")

        return []

    def _save_question(self, q: dict):
        """Save question to database."""
        db.conn.execute(
            """INSERT INTO questions
            (id, lecture_id, text, bloom, qtype, options, correct, explanation, difficulty, created_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
            (
                q.get("id"), q.get("lecture_id"), q.get("text"),
                q.get("bloom_level"), q.get("qtype"),
                json.dumps(q.get("options", [])),
                q.get("correct_answer"), q.get("explanation"),
                q.get("difficulty", 0.5), q.get("created_at")
            )
        )
        db.conn.commit()

    def process_lecture(
        self,
        title: str,
        course: str,
        content: str,
        file_type: str = "text",
        original_filename: str = ""
    ) -> dict:
        """
        * Split text into overlapping semantic chunks
        * Generate embeddings for each chunk
        * Store chunks in SQLite
        * Store vectors in Supabase
        * Return a short summary
        """
        if not title.strip():
            raise HTTPException(400, "Title required")
        if len(content.strip()) < 50:
            raise HTTPException(400, "Content too short")

        lecture_id = uuid.uuid4().hex[:16]

        def _smart_chunk(text, chunk_size=800, overlap=150):
            sentences = re.split(r"(?<=[.!?])\s+", text)
            chunks, cur = [], ""

            for s in sentences:
                s = s.strip()
                if not s:
                    continue

                if len(cur) + len(s) < chunk_size:
                    cur = (cur + " " + s).strip() if cur else s
                else:
                    if cur:
                        chunks.append(cur.strip())
                    words = cur.split()
                    cur = (
                        " ".join(words[-int(overlap / 5):]) + " " + s
                        if len(words) > int(overlap / 5)
                        else s
                    )

            if cur:
                chunks.append(cur.strip())

            return chunks if chunks else [text[:chunk_size]]

        chunks = _smart_chunk(content, 800, 150)

        embeddings = []
        for i, ch in enumerate(chunks):
            emb = self._embed(ch)
            embeddings.append({
                "chunk_idx": i,
                "text": ch,
                "faiss_idx": i,
                "embedding": emb,
                "metadata": {
                    "lecture_id": lecture_id,
                    "title": title,
                    "course": course
                }
            })

        # FIX: Pass full 'embeddings' list to save_lecture_chunks so Supabase gets the vectors
        db.save_lecture_chunks(lecture_id, embeddings)

        if self.vector_db and self.vector_db._connected:
            ok = self.vector_db.store_embeddings_batch(lecture_id, embeddings)
            print(f"✅ Supabase vector insert: {ok}")

        db.save_lecture({
            "id": lecture_id,
            "title": title,
            "course": course,
            "week": None,
            "content": content[:50000],
            "topics": [],
            "outcomes": [],
            "chunk_count": len(chunks),
            "uploaded_at": datetime.datetime.now().isoformat(),
            "file_type": file_type,
            "original_filename": original_filename
        })

        self.lecture_chunks[lecture_id] = chunks
        self.lecture_meta[lecture_id] = {
            "title": title,
            "course": course,
            "topics": []
        }

        return {
            "id": lecture_id,
            "title": title,
            "course": course,
            "chunk_count": len(chunks),
            "chunks": [{"idx": i, "preview": c[:120] + "..."} for i, c in enumerate(chunks[:5])],
            "topics": []
        }

    def summarize_lecture(self, lecture_id: str, style: str = "bullets") -> dict:
        lec = db.get_lecture_detail(lecture_id)
        if not lec:
            raise HTTPException(404, "Lecture not found")

        content = lec.get("content", "")
        if not content:
            return {"summary": "No content", "title": lec["title"]}

        key_chunks = self.search_lecture(lecture_id, "main concepts", top_k=3)
        ctx = "\n\n".join([c["text"] for c in key_chunks]) if key_chunks else content[:3000]

        prompts = {
            "bullets": "Summarise the lecture in 4-6 concise bullet points.",
            "narrative": "Write a short 2-paragraph narrative summary of the lecture.",
            "key_concepts": "Identify and briefly explain the 5 most important concepts."
        }
        system = prompts.get(style, prompts["bullets"])

        answer = self.llm.generate_chat(
            [
                {"role": "system", "content": system},
                {"role": "user", "content": f"Lecture title: {lec['title']}\n\n{ctx}"}
            ],
            max_new_tokens=500,
            temperature=0.3
        )

        return {
            "summary": answer.strip(),
            "title": lec["title"],
            "style": style
        }

# Initialise the RAG engine (uses the shared LLM & vector DB from `db`)
print("\n🔧 Initialising RAG engine …")
rag_ai = SupabaseRAGEngine(llm_adapter=shared_llm, vector_db=db.vector_db)
print(f"✅ RAG engine ready – {len(rag_ai.lecture_chunks)} cached lectures")



🔧 Initialising RAG engine …
🔹 Loaded 0 lectures from SQLite
✅ RAG engine ready – 0 cached lectures


# 🎙️ LiveLecturerAI

In [8]:
import asyncio
import time
from collections import deque

import numpy as np
import torch
import webrtcvad

from faster_whisper import WhisperModel


class LiveLecturerAI:
    """Handles audio capture, Whisper transcription and non-blocking storage."""
    def __init__(self, model_size: str = "small"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model_size = model_size
        self.stt: WhisperModel | None = None
        self.stt_loaded = False

        # Audio config
        self.sr = 16000
        self.vad = webrtcvad.Vad(2)

        # Ring buffer - keeps the last 30 seconds
        self.max_buffer_seconds = 30
        self.audio_buf = deque(maxlen=self.sr * self.max_buffer_seconds)

        # Streaming control
        self.stt_interval = 1.0
        self.min_audio_seconds = 2.0
        self.chunk_seconds = 4.0

        self._last_stt_time = 0.0
        self._lock = asyncio.Lock()

        # Session state
        self.session_id: str | None = None
        self.session_title = "Live Lecture"
        self.session_course = "General"
        self.session_start: float | None = None

        # Transcript cache
        self.transcript_chunks: list[dict] = []
        self.last_emitted_text = ""

        # Async queue for background processing
        self.chunk_queue: asyncio.Queue = asyncio.Queue(maxsize=100)
        self._consumer_task = None

    def start_background_worker(self):
        """Start the async background consumer when an event loop is running."""
        if self._consumer_task is None or self._consumer_task.done():
            self._consumer_task = asyncio.create_task(self._background_consumer())

    def load_models(self):
        if self.stt_loaded:
            return
        dev = "cuda" if self.device == "cuda" else "cpu"
        compute = "float16" if dev == "cuda" else "int8"
        print(f"🔊 Loading Whisper ({self.model_size}) on {dev} ({compute})...")
        self.stt = WhisperModel(self.model_size, device=dev, compute_type=compute)
        self.stt_loaded = True
        print("✅ Whisper ready")

    def start_session(self, title: str = "Live Lecture", course: str = "General"):
        self.session_id = f"live_{int(time.time())}"
        self.session_title = title
        self.session_course = course
        self.session_start = time.time()
        self.audio_buf.clear()
        self.transcript_chunks.clear()
        self.last_emitted_text = ""
        self.start_background_worker()
        print(f"🔴 Session started: {self.session_id} ({title})")
        return self.session_id

    def stop_session(self):
        print("🛑 Session stopped")
        self.session_id = None

    def _push_audio(self, a16: np.ndarray):
        """Append new PCM-16 floats to the ring buffer."""
        self.audio_buf.extend(a16.tolist())

    def _has_speech(self, samples: np.ndarray) -> bool:
        """VAD - cheap heuristic for silence detection."""
        if samples.size < 800:
            return False
        pcm = (np.clip(samples[:800], -1, 1) * 32767).astype(np.int16).tobytes()
        try:
            return self.vad.is_speech(pcm, self.sr)
        except Exception:
            rms = float(np.sqrt(np.mean(samples[:800] ** 2)))
            return rms > 0.02

    async def process_audio_chunk_float32(self, audio16k: np.ndarray) -> dict:
        """Core STT -> returns a dict with the transcribed text (or empty)."""
        if not self.session_id:
            return {}
        if not self.stt_loaded:
            self.load_models()
        if audio16k is None or audio16k.size == 0:
            return {}

        self._push_audio(audio16k)

        now = time.time()
        if now - self._last_stt_time < self.stt_interval:
            return {}
        self._last_stt_time = now

        if len(self.audio_buf) < int(self.min_audio_seconds * self.sr):
            return {}

        chunk_len = int(self.chunk_seconds * self.sr)
        audio_np = np.array(list(self.audio_buf)[-chunk_len:], dtype=np.float32)

        if not self._has_speech(audio_np):
            return {}

        async with self._lock:
            def _transcribe():
                segs, _ = self.stt.transcribe(
                    audio_np,
                    language="en",
                    beam_size=3,
                    vad_filter=True
                )
                return " ".join(s.text.strip() for s in segs).strip()

            text = await asyncio.to_thread(_transcribe)

        if not text or len(text) < 2:
            return {}

        chunk = {
            "session_id": self.session_id,
            "text": text,
            "timestamp": time.time()
        }
        await self.chunk_queue.put(chunk)

        self.transcript_chunks.append(chunk)
        recent = self.transcript_chunks[-20:]
        combined = " ".join(c["text"] for c in recent)

        if combined != self.last_emitted_text:
            self.last_emitted_text = combined
            return {
                "session_id": self.session_id,
                "title": self.session_title,
                "course": self.session_course,
                "transcript": combined,
                "ts": time.time()
            }

        return {}

    async def _background_consumer(self):
        """
        Runs forever, pulling raw chunks from self.chunk_queue,
        persisting them to SQLite, creating embeddings, and pushing
        them to Supabase vectors.
        """
        while True:
            chunk = await self.chunk_queue.get()
            try:
                db.conn.execute(
                    """
                    INSERT INTO lecture_chunks_local (lecture_id, chunk_idx, text, faiss_idx)
                    VALUES (?,?,?,?)
                    """,
                    (chunk["session_id"], int(chunk["timestamp"] * 1000), chunk["text"], 0)
                )
                db.conn.commit()

                emb = rag_ai._embed(chunk["text"])

                if rag_ai.vector_db and rag_ai.vector_db._connected:
                    rag_ai.vector_db.store_embeddings_batch(
                        chunk["session_id"],
                        [{
                            "chunk_idx": int(chunk["timestamp"] * 1000),
                            "text": chunk["text"],
                            "embedding": emb,
                            "metadata": {"source": "live_stream"}
                        }]
                    )
            except Exception as e:
                print(f"⚠️ Background worker error: {e}")
            finally:
                self.chunk_queue.task_done()

    async def process_stream_chunk(self, audio_tuple, _, __):
        """Compatibility shim - the socket code expects this name."""
        sr, audio_array = audio_tuple
        return await self.process_audio_chunk_float32(audio_array)


print("\n🚀 Initializing LiveLecturerAI...")
live_ai = LiveLecturerAI(model_size="small")
print("✅ LiveLecturerAI ready")



🚀 Initializing LiveLecturerAI...
✅ LiveLecturerAI ready


# 📡 FastAPI + Socket.IO


In [9]:
import asyncio, datetime, logging, time, uuid, bcrypt, socketio, nest_asyncio, numpy as np, torch
import jwt as pyjwt
from pathlib import Path
from typing import Optional
from fastapi import Depends, FastAPI, File, Form, Header, HTTPException, UploadFile, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field

nest_asyncio.apply()

# 1. Initialize FastAPI
app = FastAPI(title="Smart Classroom AI API", version="1.0.0", docs_url="/docs")
JWT_SECRET = os.getenv("JWT_SECRET", "smart-classroom-secret-2025")


# 2. Correct CORS for Vercel & ngrok
VERCEL_URL = "https://smart-classroom-ui.vercel.app/"
app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        VERCEL_URL.strip("/"),
        "http://localhost:3000",
        "http://127.0.0.1:3000",
    ],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 3. Socket.IO Setup
sio = socketio.AsyncServer(async_mode="asgi", cors_allowed_origins="*", max_http_buffer_size=10_000_000)
socket_app = socketio.ASGIApp(sio, other_asgi_app=app)

# --- Pydantic Models ---
class LoginReq(BaseModel):
    username: str
    password: str

class RegisterReq(BaseModel):
    username: str = Field(..., min_length=3)
    password: str = Field(..., min_length=6)
    email: Optional[str] = None
    role: str = "teacher"

class UploadLectureReq(BaseModel):
    title: str = Field(..., min_length=1)
    course: str = "General"
    content: str = Field(..., min_length=10)

class AskReq(BaseModel):
    question: str = Field(..., min_length=1)
    lecture_id: Optional[str] = None
    mode: str = "rag"

# --- Auth Logic ---
def _jwt_create(user):
    payload = {
        "sub": user["id"], "username": user["username"], "role": user["role"],
        "iat": int(time.time()), "exp": int(time.time() + 172800)
    }
    return pyjwt.encode(payload, JWT_SECRET, algorithm="HS256")

def _jwt_decode(token):
    return pyjwt.decode(token, JWT_SECRET, algorithms=["HS256"])

async def get_current_user(authorization: str = Header(default=None)):
    if not authorization or not authorization.lower().startswith("bearer "):
        raise HTTPException(401, "Missing Bearer token")
    try:
        payload = _jwt_decode(authorization.split(" ")[1])
        user = db.get_user_by_username(payload.get("username"))
        if not user: raise Exception()
        return user
    except: raise HTTPException(401, "Invalid token")

# --- ALL REQUIRED ROUTES ---
@app.get("/api/health")
async def health():
    return {"status": "healthy", "server": "Colab", "vector_db": db.vector_db.get_stats()}

@app.post("/api/auth/login")
async def login(data: LoginReq):
    user = db.get_user_by_username(data.username)
    if not user or not bcrypt.checkpw(data.password.encode(), user["password_hash"].encode()):
        raise HTTPException(401, "Invalid credentials")
    return {"token": _jwt_create(user), "user": {"id": user["id"], "username": user["username"], "email": user.get("email"), "role": user["role"]}}

@app.post("/api/auth/register")
async def register(data: RegisterReq):
    try:
        user = db.register_user(data.username, data.password, data.email, data.role)
        return {"token": _jwt_create(user), "user": {"id": user["id"], "username": user["username"], "email": user.get("email"), "role": user["role"]}}
    except ValueError as e: raise HTTPException(400, str(e))

@app.get("/api/auth/me")
async def get_me(current_user: dict = Depends(get_current_user)):
    return {"user": current_user}

@app.post("/api/auth/password-reset")
async def password_reset(email: str):
    return {"message": "If this email exists, reset instructions have been sent."}

@app.get("/api/lectures")
async def get_lectures(current_user: dict = Depends(get_current_user)):
    return {"lectures": db.list_lectures()}

@app.post("/api/lectures/upload")
async def upload_lecture(data: UploadLectureReq, current_user: dict = Depends(get_current_user)):
    return rag_ai.process_lecture(data.title, data.course, data.content)

@app.post("/api/lectures/upload-file")
async def upload_lecture_file(file: UploadFile = File(...), title: str = Form(...), course: str = Form("General"), current_user: dict = Depends(get_current_user)):
    content = (await file.read()).decode("utf-8")
    return rag_ai.process_lecture(title, course, content, file_type=file.content_type, original_filename=file.filename)

@app.post("/api/assistant/ask")
async def ask_assistant(data: AskReq, current_user: dict = Depends(get_current_user)):
    if data.mode == "rag":
        res = rag_ai.ask_with_context(data.question, data.lecture_id)
        return {"answer": res["answer"], "sources": res["sources"]}
    return {"answer": shared_llm.generate_chat([{"role": "user", "content": data.question}]), "sources": []}

print("✅ Backend Fully Compliant: All required API routes are live.")

✅ Backend Fully Compliant: All required API routes are live.


In [3]:
from fastapi.middleware.cors import CORSMiddleware

# 1. Update your Vercel URL here
VERCEL_URL = "https://smart-classroom-ui.vercel.app/"

# 2. Note on CORS Middleware
# To avoid RuntimeError, ensure CORS is handled inside the app initialization cell.

print(f"Backend configured for: {VERCEL_URL}")
print("If you still see a RuntimeError, move the add_middleware call to the cell where 'app' is first defined.")

Backend configured for: https://smart-classroom-ui.vercel.app/
If you still see a RuntimeError, move the add_middleware call to the cell where 'app' is first defined.


#🌐LAUNCH SERVER

In [4]:
import os
import socketio
import signal
import sys
import threading
import time
import asyncio
import uvicorn
from pyngrok import ngrok
from google.colab import userdata

# 1. Clean up port
print("Cleaning up port 8000...")
!fuser -k 8000/tcp 2>/dev/null || true
time.sleep(2)

PORT = 8000

# 2. Setup ngrok
try:
    ngrok.kill()
except:
    pass

public_url = None

if "socket_app" not in globals():
    if "sio" in globals() and "app" in globals():
        print("⚠️ socket_app not defined; recreating from existing app and sio objects.")
        socket_app = socketio.ASGIApp(sio, other_asgi_app=app)
    else:
        raise RuntimeError("socket_app undefined: rerun the backend setup cell before launching the server")

NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")

if NGROK_AUTH_TOKEN and len(NGROK_AUTH_TOKEN) > 20:
    try:
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        tunnel = ngrok.connect(PORT, bind_tls=True)
        public_url = tunnel.public_url
        print(f"\nBACKEND URL: {public_url}")
    except Exception as e:
        print(f"ngrok failed: {e}")

# 3. Server Runner (Fixed asyncio issue)
async def run_server():
    config = uvicorn.Config(
        app=socket_app,
        host="0.0.0.0",
        port=PORT,
        log_level="info",
        access_log=False
    )
    server = uvicorn.Server(config)
    await server.serve()

def start_background_loop():
    try:
        asyncio.run(run_server())
    except Exception as e:
        print(f"Server loop error: {e}")

# Start server
print("\nLaunching Smart Classroom Backend...")
server_thread = threading.Thread(target=start_background_loop, daemon=True)
server_thread.start()
time.sleep(5)

if public_url:
    print("="*70)
    print("SMART CLASSROOM BACKEND IS ONLINE")
    print("="*70)
    print(f"FRONTEND UI:  {VERCEL_URL}")
    print(f"BACKEND URL:   {public_url}")
    print(f"API DOCS:      {public_url}/docs")
    print("="*70)
    print("\nINSTRUCTIONS:")
    print("="*70)
else:
    print("\n!!! NGROK_AUTH_TOKEN not found. Use local port 8000. !!!")

Cleaning up port 8000...

BACKEND URL: https://intersystematic-yolonda-gymnogenous.ngrok-free.dev

Launching Smart Classroom Backend...
Server loop error: name 'socket_app' is not defined
SMART CLASSROOM BACKEND IS ONLINE
FRONTEND UI:  https://smart-classroom-ui.vercel.app/
BACKEND URL:   https://intersystematic-yolonda-gymnogenous.ngrok-free.dev
API DOCS:      https://intersystematic-yolonda-gymnogenous.ngrok-free.dev/docs

INSTRUCTIONS:
